# All-MoE Induction Eval

Follow-on to the decode/cot/moe **archetype** study in `notebooks/periodic/induction_eval.ipynb`. That experiment controls for reasoning *archetype* (dense-non-reasoning / dense-reasoning / non-reasoning-MoE). Having established those effects, this notebook holds architecture class **constant -- all three models are Mixture-of-Experts** -- and compares three modern open-weight MoE generalists head-to-head on the *identical* periodic induction quizzes (same template, `PeriodicConfig(n=9, labels=9)`, `BASE_SEED=1776`), so results line up directly against the archetype study.

**Trio (all MoE):** `qwen3.5-397b-a17b` (397B/17B active), `nemotron-3-super-120b-a12b` (120B/12B), `gpt-oss-120b` (120B/5.1B) -- the strongest open-weight generalists with a measured Lean 4 miniF2F number in the UW study (arXiv 2606.05632). All fit one p5 (640 GB) at tp=8.

**⚠️ Before a LIVE run:**
1. **Nemotron repo unverified** -- `nvidia/Nemotron-3-Super-120B-A12B` 401'd unauthenticated (likely gated). Confirm the exact repo in its `EC2_DEPLOY_SPECS` entry (`smolbench/evals/ec2.py`); if gated, set a real `HF_TOKEN` in this notebook's `keys.env` and accept the license.
2. **Precision is heterogeneous** (Qwen FP8 / GPT-OSS MXFP4 / Nemotron BF16-or-FP8) -- unavoidable since GPT-OSS ships MXFP4-only. Note it as a cross-model confound (the periodic trio was uniform FP8).
3. **Reasoning toggles differ per model** -- all three can reason; each run cell requests `max_completion_tokens=8192` for headroom, but confirm each model's thinking-mode mechanism (Qwen hybrid template / GPT-OSS reasoning-effort / Nemotron system-prompt) before trusting the contrast.
4. **Power** -- the sibling's `power_analysis.py` was sized for the 3-archetype contrast; re-derive R for this 3-MoE design (R=30 is carried over as a placeholder).

All lifecycle/harness plumbing lives in `smolbench.induction.experiment.InductionExperiment`; see the sibling notebook's intro for the full replicate / resume / cost mechanics. **Cost:** provisioning spins up one large EC2 spot instance (~$30-45/h for the p5e/p5 family) with an idle watchdog + max-lifetime backstop -- provision right before running, and tear down at the end.

**keys.env:** the first executable statement below MUST stay `load_dotenv(...)` -- `smolbench.evals.ec2` captures its `EC2_*` config (here `EC2_EXPERIMENT_TAG=periodic-moe-induction`) from `os.environ` at import time, so keys.env must land before that module is ever imported.

In [ ]:
"""The following generates the Quiz all our models will be evaluated on."""

import string

import logging

from dotenv import load_dotenv
from pathlib import Path

logging.basicConfig(level=logging.INFO)
load_dotenv(Path.cwd() / "keys.env", verbose=True)

from smolbench.induction.periodic import (
    PeriodicConfig,
    Prompter,
    get_periodic_numeric_quiz,
    numeric_count_query_gen,
)

# Per-model names: keys of EC2_DEPLOY_SPECS in smolbench/evals/ec2.py (each
# is also vLLM's --served-model-name, sent in the OpenAI request body
# verbatim). This study holds the architecture class constant -- all three
# are Mixture-of-Experts -- so the tags name the MODEL, not a decode/cot/moe
# archetype. Qwen3.5 (Qwen/...-FP8) and GPT-OSS (openai/...) are UNGATED;
# Nemotron-3-Super may be gated -- see its ec2.py TODO and keys.env HF_TOKEN.
MODEL_QWEN     = "qwen3.5-397b-a17b"           # Qwen/Qwen3.5-397B-A17B-FP8 (MoE 397B/17B, FP8)
MODEL_NEMOTRON = "nemotron-3-super-120b-a12b"  # nvidia/Nemotron-3-Super-120B-A12B (MoE 120B/12B) -- repo UNVERIFIED
MODEL_GPTOSS   = "gpt-oss-120b"                # openai/gpt-oss-120b (MoE 120B/5.1B, MXFP4)

template = string.Template(
    "You are a precise integer counter.\n"
    "\n"
    "Task: answer the question below with a single integer and nothing else.\n"
    "\n"
    "Output format:\n"
    "Return exactly one integer and nothing else.\n"
    "Do not output any explanation, punctuation, quotes, or extra whitespace.\n"
    "Stop immediately after writing the integer.\n"
    "\n"
    "Context:\n"
    "There is a counting game. Positions are counted starting from 1. "
    "At each position, words are written according to the following rules:\n"
    "$positive_info\n"
    "Question:\n"
    "How many of the positions 1 through $seq_len include '$label'?"
)

# --- Replication setup -----------------------------------------------------
# Identical quizzes to notebooks/periodic (same template, n=9, BASE_SEED) so
# this all-MoE study is directly comparable to the archetype study. R=30 is
# carried over as a placeholder -- re-derive it for the 3-MoE contrast.
BASE_SEED: int = 1776  # seed of the original preliminary run == replicate 0
INFO_TYPES: tuple[str, ...] = ("intens", "extens", "noise_intens")


def make_quizzes(seed: int) -> dict[str, tuple]:
    """Generates one replicate's three info-type quizzes, keyed by info type."""
    return dict(
        zip(
            INFO_TYPES,
            get_periodic_numeric_quiz(
                PeriodicConfig(
                    n=9,
                    labels=9,
                    seed=seed,
                ),
                Prompter(
                    template,
                    {},
                    numeric_count_query_gen,
                ),
            ),
        )
    )


# First-replicate aliases for the Prompt Validation cells below.
_base_quizzes: dict[str, tuple] = make_quizzes(BASE_SEED)
intens_quiz = _base_quizzes["intens"]
extens_quiz = _base_quizzes["extens"]
noise_intens_quiz = _base_quizzes["noise_intens"]


In [ ]:
# Builds this notebook's InductionExperiment: the replicate harness
# (results/{tag}_{info}/rep_{seed}.yaml, serialized right after grading) +
# the EC2 spot-instance lifecycle. See smolbench/evals/replicates.py and
# smolbench/induction/experiment.py.
from smolbench.induction.experiment import InductionExperiment

EXPERIMENT = InductionExperiment(
    notebook_dir="periodic_moe",
    archetype_tags={MODEL_QWEN: "qwen35", MODEL_NEMOTRON: "nemotron3", MODEL_GPTOSS: "gptoss"},
    make_quizzes=make_quizzes,
    n_replicates=30,
    base_seed=BASE_SEED,
    # Private EC2 state file so this study's instance record never clobbers
    # the periodic experiment's default .ec2_state.json (chromatic isolates
    # itself the same way). Pairs with EC2_EXPERIMENT_TAG in keys.env.
    state_file=".ec2_state_periodic_moe.json",
)


In [ ]:
# Provisions (or reattaches to) this experiment's EC2 spot instance --
# idempotent via the state file / smolbench:experiment tag, so it survives
# kernel restarts. Live AWS call; see the intro's cost note.
state = EXPERIMENT.provision()


## Prompt Validation

In [ ]:
print(intens_quiz[0].prompt)

In [ ]:
print(extens_quiz[0].prompt)

In [ ]:
print(noise_intens_quiz[0].prompt)

## Qwen3.5-397B-A17B (MoE)
`Qwen/Qwen3.5-397B-A17B-FP8` -- 397B total / 17B active, FP8, Apache-2.0, ungated. ~397 GB at FP8 -> fits one p5 (640 GB) at tp=8 with ~243 GB KV headroom. Hybrid thinking model; the run cell allows 8192 completion tokens for reasoning headroom. The swap waits on the checkpoint download/load the first time; reruns hit the instance's warm HF cache.

In [ ]:
# Swaps the shared instance's vLLM to Qwen3.5 and runs every outstanding
# replicate across all info types; finished replicates are skipped on
# rerun. If you want a controlled reasoning-on/off contrast, set Qwen's
# thinking mode here (hybrid template flag) -- default mode is used as-is.
EXPERIMENT.run(MODEL_QWEN, extra_args={"max_completion_tokens": 8192})


In [ ]:
EXPERIMENT.summarize(MODEL_QWEN)

## Nemotron-3-Super-120B-A12B (MoE)
`nvidia/Nemotron-3-Super-120B-A12B` -- 120B total / 12B active. **⚠️ repo id UNVERIFIED** (401'd unauthenticated; likely gated). Confirm the exact repo in `smolbench/evals/ec2.py` and set `HF_TOKEN` in `keys.env` (+ accept the license) before this cell will serve. Nemotron models typically toggle reasoning via a system prompt (cf. `nemotron-ultra-253b`'s `"detailed thinking on"`) -- if so, add `system_prompt` to its `EC2_DEPLOY_SPECS` entry so user prompts stay byte-identical across the trio.

In [ ]:
# Swaps vLLM to Nemotron-3-Super. Will fail fast until its repo id /
# HF_TOKEN are confirmed (see ec2.py TODO + keys.env). 8192 completion
# tokens for reasoning headroom.
EXPERIMENT.run(MODEL_NEMOTRON, extra_args={"max_completion_tokens": 8192})


In [ ]:
EXPERIMENT.summarize(MODEL_NEMOTRON)

## GPT-OSS-120B (MoE)
`openai/gpt-oss-120b` -- 120B total / 5.1B active, native MXFP4, Apache-2.0, ungated. ~63 GB -> trivially fits p5 at tp=8. Trained on the harmony response format (vLLM's chat template applies it); has low/medium/high reasoning-effort levels (default medium) -- the run cell allows 8192 completion tokens.

In [ ]:
# Swaps vLLM to GPT-OSS-120B. Reasoning effort defaults to medium; set it
# explicitly in extra_args if you want a controlled level across the trio.
EXPERIMENT.run(MODEL_GPTOSS, extra_args={"max_completion_tokens": 8192})


In [ ]:
EXPERIMENT.summarize(MODEL_GPTOSS)

# Teardown

In [ ]:
# Terminates the spot instance (and its EBS volume) and clears the private
# .ec2_state_periodic_moe.json. Also works after a kernel restart / lost
# state file: falls back to the smolbench:experiment=periodic-moe-induction tag.
EXPERIMENT.teardown()
